In [ ]:
import pdfplumber
import pandas as pd

def table_to_markdown(table_data):
    """แปลงตาราง raw list เป็น Markdown table string"""
    # กรองแถวที่ว่างเปล่าออก
    clean_table = [[cell if cell is not None else "" for cell in row] for row in table_data if any(row)]
    if not clean_table:
        return ""
    
    # ใช้แถวแรกเป็น Header
    headers = clean_table[0]
    df = pd.DataFrame(clean_table[1:], columns=headers)
    return df.to_markdown(index=False)

def extract_durian_pdf(pdf_path):
    extracted_documents = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            page_num = page_idx + 1
            
            # 1. ค้นหา Bounding Boxes ของตารางทั้งหมดในหน้านี้
            find_tables = page.find_tables()
            table_bboxes = [table.bbox for table in find_tables]
            
            # 2. Condition: สกัดตาราง (ถ้ามี)
            extracted_tables = []
            
            if find_tables:
                for table in find_tables:
                    raw_table = table.extract()
                    md_table = table_to_markdown(raw_table)
                    if md_table:
                        extracted_tables.append(md_table)
                       

            # 3. Condition: สกัดข้อความทั่วไปที่ไม่ทับซ้อนกับพื้นที่ตาราง
            # กรองพื้นที่ที่เป็นตารางออกก่อนดึง text
            def not_inside_tables(obj):
                x0, top, x1, bottom = obj["x0"], obj["top"], obj["x1"], obj["bottom"]
                for (tx0, ttop, tx1, tbottom) in table_bboxes:
                    # ถ้าอักขระอยู่ในกรอบตาราง ให้กรองทิ้ง
                    if not (x1 < tx0 or x0 > tx1 or bottom < ttop or top > tbottom):
                        return False
                return True

            page_text = page.filter(not_inside_tables).extract_text(layout=False) or ""

            # 4. รวมข้อมูลพร้อมจัด Metadata ประจำหน้า
            content_blocks = []
            
            if page_text.strip():
                content_blocks.append(f"### [เนื้อหาทั่วไป]\n{page_text.strip()}")
            
            if extracted_tables:
                for idx, tbl in enumerate(extracted_tables, 1):
                    content_blocks.append(f"### [ตารางข้อมูลที่ {idx}]\n{tbl}")

            full_page_content = "\n\n".join(content_blocks)
            
            extracted_documents.append({
                "page": page_num,
                "content": full_page_content,
                "metadata": {
                    "source": pdf_path,
                    "page": page_num,
                    "has_table": len(extracted_tables) > 0,
                    "category": "ข้อมูลโรคและการรักษาทุเรียน"
                }
            })

    return extracted_documents

In [4]:
extract_durian_pdf(r"C:\Users\msapi\OneDrive\Documents\RichProject\RAGDurian\durian.pdf")

[{'page': 1,
  'content': '### [เนื้อหาทั่วไป]\nโรคของทุเรียน\nโรครากเน่าและโคนเน่าของทุเรียน (Root and Foot Rot)\nสาเหต ุ เชื้อราไฟทอฟธอรา (Phytophthora palmovora (Butler) Butler)\nลักษณะอาการ\nใบจะไม่เป็นมันสดใสเหมือนใบทุเรียนปกติ ต่อมาใบล่างๆ จะเริ่มเป็นจุดประเหลืองแล้วค่อยๆ หลุด\nร่วงไป ต้นทรุดโทรมและตาย\nเกิดอาการเน่าที่โคนต้นหรือกิ่ง จะสังเกตเห็นผิวเปลือกของลำต้นหรือกิ่งคล้ายมีคราบน้ำเกาะติด\nเห็นได้ชัดในสภาพที่ต้นทุเรียนแห้ง ในช่วงเช้าที่มีอากาศชุ่มชื้นจะมองเห็นหยดน้ำยางสีน้ำตาลแดงไหล\nออกมาจากรอยแผลแตกของลำต้นหรือกิ่ง และน้ำยางนี้จะค่อยๆ แห้งไปในช่วงกลางวันที่มีแดดจัด ทำให้\nเห็นเป็นคราบน้ำจับบนเปลือกของลำต้น เมื่อถากเปลือกของลำต้นบริเวณที่มีคราบน้ำยาง จะเห็นเนื้อเยื่อ\nเปลือกถูกทำลายมีสีน้ำตาลแดง หรือน้ำตาลเข้ม ส่วนอาการเน่าที่เกิดกับรากเล็กหรือรากฝอยนั้น เนื้อเยื่อ\nรากจะเปื่อยยุ่ย เมื่อดึงเบาๆ จะขาดออกจากกันได้ง่าย\nการแพร่ระบาด\nเชื้อราไฟทอฟธอราสามารถพักตัวอยู่ในดินได้เป็นเวลานานหลายปี ในรูปแบบของคลาไมโดสปอร์\n(chlamydospores) และเมื่อสภาวะแวดล้อมเหมาะสม คือน้ำและความชื้นเพ